# Pisto GPT 64M - Kaggle Fine-tuning
Self-contained instruction-tuning notebook (no Google Drive needed).

## One-time setup
1. **GitHub secret** (only if your repo is private): add a Kaggle Secret named `GH_TOKEN` with a GitHub PAT. Skip if public.
2. **Upload pretrained weights**: create a Kaggle Dataset (e.g. `pisto-weights`) and upload your `pretrain_best.pt` into it. The notebook auto-detects it at `/kaggle/input/<dataset>/pretrain_best.pt`.
3. Run all cells. `instruct_best.pt` is saved to `/kaggle/working/pg/weights/` and downloadable from the notebook **Output** tab.

The notebook retrains the Arabic tokenizer deterministically (so it matches your pretrained checkpoint) and applies the tuned finetune config (lower LR, more CIDAR data, less manual-repeat, stronger dropout).

In [ ]:
!pip install -q torch datasets tokenizers
print('deps installed')

In [ ]:
import os
if not os.path.exists('/kaggle/working/pg/config/instruct.json'):
    !git clone https://github.com/BayanDrp/pisto-gpt-64m.git /kaggle/working/pg
    print('cloned')
else:
    print('already present')
print('repo ready:', os.path.exists('/kaggle/working/pg/config/instruct.json'))

In [ ]:
import json, os
cfg_path = '/kaggle/working/pg/config/instruct.json'
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['model']['dropout'] = 0.1
cfg['training']['lr'] = 3e-5
cfg['training']['max_hours'] = 4
cfg['training']['max_steps'] = 8000
cfg['dataset']['alpaca_max'] = 8000
cfg['dataset']['manual_repeat'] = 2
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)
print('config patched -> lr=3e-5  alpaca_max=8000  manual_repeat=2  dropout=0.1')

In [ ]:
import json, os
from datasets import load_dataset
ds = load_dataset('Jr23xd23/ArabicText-Large', split='train', streaming=True)
texts = []
for i, s in enumerate(ds):
    if i >= 100_000:
        break
    t = s.get('text', '')
    if len(t) > 50:
        texts.append(t)
print(f'Got {len(texts):,} Arabic docs for tokenizer training')
tok_data = '/kaggle/working/pg/tmp_arabic.txt'
with open(tok_data, 'w') as f:
    for t in texts:
        f.write(t.replace(chr(10), ' ') + chr(10))
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer = trainers.BpeTrainer(vocab_size=8192, special_tokens=['<PAD>', '<BOS>', '<EOS>'], min_frequency=2)
tok.train([tok_data], trainer)
tok_path = '/kaggle/working/pg/config/bpe_tokenizer.json'
tok.save(tok_path)
print(f'Trained Arabic tokenizer -> {tok_path} (vocab {tok.get_vocab_size()})')

In [ ]:
import glob, os, shutil
WORK = '/kaggle/working/pg'
os.makedirs(f'{WORK}/weights', exist_ok=True)
cands = glob.glob('/kaggle/input/*/pretrain_best.pt') + glob.glob('/kaggle/input/*/*/pretrain_best.pt')
if cands:
    shutil.copy(cands[0], f'{WORK}/weights/pretrain_best.pt')
    print(f'Copied pretrained weights from {cands[0]}')
else:
    print('WARNING: pretrain_best.pt not found in /kaggle/input -> finetune will start from scratch')

In [ ]:
import subprocess, sys
WORK = '/kaggle/working/pg'
print('Starting fine-tuning (auto-stops on overfit)...')
proc = subprocess.run([sys.executable, 'training/finetune.py'], cwd=WORK)
print('Fine-tuning exit code:', proc.returncode)

In [ ]:
import glob, os
best = glob.glob('/kaggle/working/pg/weights/instruct_best.pt')
print('instruct_best.pt saved:', bool(best))
if best:
    print('Path:', best[0])
    print('Download it from the notebook Output tab (pg/weights/instruct_best.pt).')